# Poisson-Process Model of Trade Arrivals

We treat the trade open-timestamps produced by the backtester as a point process and ask: are arrivals consistent with a homogeneous Poisson process? If so, inter-arrival times should be approximately exponential with rate `lambda = N / T`.

**Procedure**
1. Run the backtester (compounding mode) and pull `bt.trades`.
2. Compute inter-arrival gaps (in days) between successive `init_timestamp`s.
3. MLE: `lambda_hat = 1 / mean_gap`.
4. Kolmogorov-Smirnov test of the gaps vs Exp(lambda_hat).
5. Overlay: histogram of gaps vs fitted exponential PDF.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from main import process_data, strat
from backtester import BackTester

data = pd.read_csv('../BTC_2019_2023_1d.csv')
result = strat(process_data(data))
result.to_csv('../updated_final_data.csv', index=False)

bt = BackTester('BTC',
                signal_data_path='../updated_final_data.csv',
                master_file_path='../updated_final_data.csv',
                compound_flag=1)
bt.get_trades(1000)
len(bt.trades)

In [ ]:
# Inter-arrival times of trade opens (in days).
opens = pd.to_datetime([t.init_timestamp for t in bt.trades]).sort_values()
gaps_days = np.diff(opens.values).astype('timedelta64[s]').astype(float) / 86400.0

lam_hat = 1.0 / gaps_days.mean()
print(f'N trades        : {len(bt.trades)}')
print(f'mean gap (days) : {gaps_days.mean():.3f}')
print(f'lambda_hat /day : {lam_hat:.4f}')
print(f'expected gaps/yr: {lam_hat * 365:.2f} trades/yr')

In [ ]:
# KS test: H0 = gaps drawn from Exp(lambda_hat).
ks_stat, ks_p = stats.kstest(gaps_days, 'expon', args=(0.0, 1.0 / lam_hat))
print(f'KS statistic : {ks_stat:.4f}')
print(f'p-value      : {ks_p:.4f}')
print('REJECT H0 (not exponential)' if ks_p < 0.05 else 'Cannot reject H0 (consistent with exponential)')

In [ ]:
xs = np.linspace(0, gaps_days.max(), 200)
pdf = lam_hat * np.exp(-lam_hat * xs)

plt.figure(figsize=(10, 4))
plt.hist(gaps_days, bins=30, density=True, alpha=0.6, label='empirical gaps')
plt.plot(xs, pdf, 'r-', label=f'Exp(lambda={lam_hat:.3f})')
plt.xlabel('inter-arrival time (days)'); plt.ylabel('density')
plt.title('Trade inter-arrival times vs fitted exponential')
plt.legend(); plt.tight_layout(); plt.show()

## Interpretation

Because the volume-spike trigger is itself volatility-clustered, we expect the gaps to deviate from a pure Poisson model -- trades will cluster during high-volume regimes and thin out otherwise. The KS p-value quantifies the deviation. A natural generalization is a Cox / doubly-stochastic Poisson process where `lambda(t)` is driven by realized volatility; that would be a separate notebook.